In [ ]:
from xc import XenoCantoDownloader
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

In [ ]:
xcd = XenoCantoDownloader(api_key=os.environ["XC_API_KEY"])

In [ ]:
query = xcd.build_query()
res = xcd.get_page(query)
res

In [ ]:
data = xcd(query="box:32.485,-117.582,33.482,-115.228")
d
#box:32.485,-117.582,33.482,-115.228

In [ ]:
import json
import requests
with open("xc_meta.json", mode="w") as f:
    json.dump(data, f, indent=4)

In [ ]:
req = requests.get(data[0]["recordings"][0]["file"])
req

In [ ]:
data[0]["recordings"][0]["file"]

In [ ]:
req.content

In [ ]:
import shutil
import os
from pathlib import Path
from multiprocessing.pool import ThreadPool

# https://stackoverflow.com/questions/16694907/download-large-file-in-python-with-requests
def download_file(url, local_filename, dry_run=False):
    if os.path.exists(local_filename):
        return local_filename

    try:
        with requests.get(url, stream=True) as r:
            with open(local_filename, 'wb') as f:
                if not dry_run:
                    shutil.copyfileobj(r.raw, f)
                else:
                    print(local_filename)

        return local_filename
    except IOError as e:
        print(e, flush=True)
        return None

def download_files(xcd, data, parent_folder="data/xeno-canto", workers = 4):
    def prep_download(args):
        url = args[0]
        file_path = args[1]
        return download_file(url, file_path)

    os.makedirs(parent_folder, exist_ok=True)

    if "recordings" in data[0]:
        data = xcd.concat_recording_data(data) 
    download_data = [
        (recording["file"], Path(parent_folder) / Path(recording["file-name"]))
        for recording in data
    ]
    pool = ThreadPool(workers)
    results = pool.imap_unordered(prep_download, download_data) 
    pool.close()
    return results

download_files(xcd, data)

# Study

In [ ]:
import pandas as pd
recordings = xcd.concat_recording_data(data)
df = pd.DataFrame(recordings)

In [ ]:
!uv add --optional notebooks seaborn

In [ ]:
import matplotlib
import seaborn as sns
# df["en"].value_counts().hist()


sns.histplot(df["en"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

plt.ylabel("Number of Species")
plt.xlabel("Number of Indivuals Per Species")
plt.title("Do We Have a Few-shot Learning Problem for XC in Southern California?")
df["en"].value_counts().hist()
